In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_00_seed (parameterized)
# Run ONCE (manual). Creates seed in landing/_seed/
# Volume/schema env config_setup se aate hain

import uuid, random, datetime as dt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

ROOT = LANDING
SEED = f"{ROOT}/_seed"

# ============================================================
# 1. Create catalog/schema/volume if not exists (env-aware)
# ============================================================
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{HEAVY_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{HEAVY_SCHEMA}.{LANDING_VOLUME}")
print(f"✅ Volume ready: {ROOT}")

# ============================================================
# 2. FALLBACK — empty existing _seed/ folder if present
# ============================================================
print("🧹 Clearing existing _seed/ (fallback)...")
try:
    files = dbutils.fs.ls(SEED)
    if files:
        dbutils.fs.rm(SEED, recurse=True)
        print(f"   ✅ Removed existing _seed/ ({len(files)} items)")
    else:
        print("   ℹ️  _seed/ already empty")
except Exception as e:
    print(f"   ℹ️  _seed/ not found or empty — fresh start ({e})")

# ============================================================
# 3. Helper — rounding without Spark clash
# ============================================================
def rnd(x, n=6):
    return float(f"{x:.{n}f}")

# ============================================================
# 4. Create seed
# ============================================================
print("🌱 Creating seed...")
random.seed(42)

# ---- machine_models (8) ----
models = [{"model_id": f"MDL-{i:02d}", "model_name": f"Model_{i}",
           "category": random.choice(["tractor","bulldozer","excavator"])}
          for i in range(1, 9)]

# ---- owners (10) ----
owners = [{"owner_id": f"OWN-{i:03d}", "owner_name": f"Owner_{i}",
           "owner_type": random.choice(["customer","dealer"]),
           "country_code": random.choice(["IN","US","DE"])}
          for i in range(1, 11)]

# ---- operators (40) — FK: owner ----
operators = [{"operator_id": f"OPR-{i:03d}", "operator_name": f"Op_{i}",
              "license_no": f"LIC{random.randint(10000,99999)}",
              "owner_id": random.choice(owners)["owner_id"]}
             for i in range(1, 41)]

# ---- sites (15) ----
sites = [{"site_id": f"SITE-{i:02d}", "site_name": f"Site_{i}",
          "country_code": random.choice(["IN","US","DE"]),
          "geo_lat": rnd(random.uniform(-30, 55), 6),
          "geo_lon": rnd(random.uniform(-120, 145), 6)}
         for i in range(1, 16)]

# ---- machines (50) — FK: model, owner, site ----
machines = [{"machine_id": f"MCH-{i:04d}",
             "model_id": random.choice(models)["model_id"],
             "owner_id": random.choice(owners)["owner_id"],
             "site_id": random.choice(sites)["site_id"],
             "status": random.choice(["active","idle","maintenance"]),
             "manufacture_year": random.randint(2015, 2024)}
            for i in range(1, 51)]

# ---- vendors (20) ----
vendors = [{"vendor_id": f"VEN-{i:03d}", "vendor_name": f"Vendor_{i}",
            "country_code": random.choice(["IN","US","DE"])}
           for i in range(1, 21)]

# ---- materials (60) ----
materials = [{"material_id": f"MAT-{i:03d}", "material_desc": f"Part_{i}",
              "uom": random.choice(["EA","KG","L"])}
             for i in range(1, 61)]

# ---- cost_centers (5) ----
ccs = [{"cost_center_id": f"CC-{i:02d}", "cost_center_name": f"CC_{i}"}
       for i in range(1, 6)]

# ---- geofences (10) — FK: site ----
geofences = [{"geofence_id": f"GEO-{i:02d}",
              "site_id": random.choice(sites)["site_id"],
              "geofence_name": f"GF_{i}"} for i in range(1, 11)]

# ============================================================
# 5. Write seed tables
# ============================================================
seed_data = {
    "machine_models": models,
    "owners": owners,
    "operators": operators,
    "sites": sites,
    "machines": machines,
    "vendors": vendors,
    "materials": materials,
    "cost_centers": ccs,
    "geofences": geofences,
}

for name, data in seed_data.items():
    spark.createDataFrame(data).write.mode("overwrite").json(f"{SEED}/{name}")
    print(f"   ✓ {name}: {len(data)} rows")

print(f"\n✅ Seed complete: {len(machines)} machines, {len(sites)} sites")
print(f"   Location: {SEED}")